### BILSTM Model 

In [5]:
import numpy as np
import pandas as pd
import pickle
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Input, Embedding, Bidirectional,LSTM, Dense,Dropout,BatchNormalization
from tensorflow.keras.optimizers import Adam, RMSprop
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score
)

### load the datasets

In [6]:
X_train_padded = np.load("models/X_train_padded.npy")
X_val_padded = np.load("models/X_val_padded.npy")
X_test_padded = np.load("models/X_test_padded.npy")
y_train = np.load("models/y_train.npy")
y_val = np.load("models/y_val.npy")
y_test = np.load("models/y_test.npy")

sequence_length = X_train_padded.shape[1]
vocab_size = 20000

print(X_train_padded.shape)
print(X_val_padded.shape)
print(X_test_padded.shape)

(34705, 200)
(7439, 200)
(7438, 200)


In [7]:
bilstm_results= []

def evaluate_bilstm(model, experiment_name, X_test_data=None):

    # Use 200-token test data by default
    if X_test_data is None:
        X_test_data = X_test_padded

    # Prediction probabilities
    y_prob = model.predict(
        X_test_data,
        verbose=0
    ).ravel()

    # Convert probabilities to classes
    y_pred = (y_prob >= 0.5).astype(int)

    # Evaluation metrics
    accuracy = accuracy_score(y_test, y_pred)
    precision = precision_score(y_test, y_pred)
    recall = recall_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)
    roc_auc = roc_auc_score(y_test, y_prob)

    # Store results
    bilstm_results.append({
        "Experiment": experiment_name,
        "Accuracy": accuracy,
        "Precision": precision,
        "Recall": recall,
        "F1 Score": f1,
        "ROC-AUC": roc_auc
    })

    # Display results
    print(f"\n{experiment_name}")
    print("-" * 40)
    print(f"Accuracy : {accuracy:.4f}")
    print(f"Precision: {precision:.4f}")
    print(f"Recall   : {recall:.4f}")
    print(f"F1 Score : {f1:.4f}")
    print(f"ROC-AUC  : {roc_auc:.4f}")

### Baseline BILSTM

In [9]:
bilstm_model = Sequential([
    Input(shape=(sequence_length,)),

    Embedding(
        input_dim=vocab_size,
        output_dim=128
    ),
    Bidirectional(
        LSTM(64)
    ),

    Dense(
        64,
        activation="relu"
    ),

    Dense(
        1,
        activation="sigmoid"
    )
])

bilstm_model.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

bilstm_model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_2 (Embedding)         │ (None, 200, 128)       │     2,560,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional_1 (Bidirectional) │ (None, 128)            │        98,816 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 1)              │            65 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,667,137 (10.17 MB)

 Trainable params: 2,667,137 (10.17 MB)

 Non-trainable params: 0 (0.00 B)

In [10]:
history_bilstm = bilstm_model.fit(
    X_train_padded,
    y_train,
    validation_data=(X_val_padded, y_val),
    epochs=5,
    batch_size=32,
    verbose=1
)

Epoch 1/5
1085/1085 ━━━━━━━━━━━━━━━━━━━━ 112s 102ms/step - accuracy: 0.8403 - loss: 0.3637 - val_accuracy: 0.8829 - val_loss: 0.2874
Epoch 2/5
1085/1085 ━━━━━━━━━━━━━━━━━━━━ 117s 107ms/step - accuracy: 0.9245 - loss: 0.2060 - val_accuracy: 0.8650 - val_loss: 0.3181
Epoch 3/5
1085/1085 ━━━━━━━━━━━━━━━━━━━━ 117s 108ms/step - accuracy: 0.9520 - loss: 0.1337 - val_accuracy: 0.8804 - val_loss: 0.3525
Epoch 4/5
1085/1085 ━━━━━━━━━━━━━━━━━━━━ 117s 108ms/step - accuracy: 0.9718 - loss: 0.0836 - val_accuracy: 0.8649 - val_loss: 0.3897
Epoch 5/5
1085/1085 ━━━━━━━━━━━━━━━━━━━━ 122s 112ms/step - accuracy: 0.9817 - loss: 0.0533 - val_accuracy: 0.8730 - val_loss: 0.4353


In [12]:
evaluate_bilstm(
    bilstm_model,
    "BILSTM - Baseline"
)
bilstm_model.save("models/BILSTM_baseline.keras")


BILSTM - Baseline
----------------------------------------
Accuracy : 0.8672
Precision: 0.8731
Recall   : 0.8604
F1 Score : 0.8667
ROC-AUC  : 0.9361


### Embedding Dimension

In [15]:
bilstm_embedding = Sequential([
    Input(shape=(sequence_length,)),

    Embedding(
        input_dim=vocab_size,
        output_dim=256
    ),

    Bidirectional(
        LSTM(64)
    ),


    Dense(
        64,
        activation="relu"
    ),

    Dense(
        1,
        activation="sigmoid"
    )
])

bilstm_embedding.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=["accuracy"]
)



In [16]:
history_embedding = bilstm_embedding.fit(
    X_train_padded,
    y_train,
    validation_data=(X_val_padded, y_val),
    epochs=5,
    batch_size=32,
    verbose=1
)

Epoch 1/5
1085/1085 ━━━━━━━━━━━━━━━━━━━━ 150s 137ms/step - accuracy: 0.8269 - loss: 0.3901 - val_accuracy: 0.8859 - val_loss: 0.2874
Epoch 2/5
1085/1085 ━━━━━━━━━━━━━━━━━━━━ 152s 140ms/step - accuracy: 0.9213 - loss: 0.2105 - val_accuracy: 0.8840 - val_loss: 0.3077
Epoch 3/5
1085/1085 ━━━━━━━━━━━━━━━━━━━━ 149s 137ms/step - accuracy: 0.9513 - loss: 0.1349 - val_accuracy: 0.8744 - val_loss: 0.3340
Epoch 4/5
1085/1085 ━━━━━━━━━━━━━━━━━━━━ 149s 137ms/step - accuracy: 0.9725 - loss: 0.0804 - val_accuracy: 0.8633 - val_loss: 0.5253
Epoch 5/5
1085/1085 ━━━━━━━━━━━━━━━━━━━━ 149s 137ms/step - accuracy: 0.9826 - loss: 0.0549 - val_accuracy: 0.8675 - val_loss: 0.4764


In [27]:
evaluate_bilstm(
    bilstm_embedding,
    "BILSTM - Embedding 256"
)

bilstm_embedding.save("models/BILSTM_embedding_256.keras")


BILSTM - Embedding 256
----------------------------------------
Accuracy : 0.8656
Precision: 0.8532
Recall   : 0.8843
F1 Score : 0.8685
ROC-AUC  : 0.9296


### Sequential Length

In [28]:
with open("models/tokenizer.pkl", "rb") as file:
    tokenizer = pickle.load(file)

vocab_size = len(tokenizer.word_index) + 1

In [29]:
X_train_text = pd.read_pickle("models/X_train_text.pkl")
X_val_text = pd.read_pickle("models/X_val_text.pkl")
X_test_text = pd.read_pickle("models/X_test_text.pkl")

print(X_train_text.shape)
print(X_val_text.shape)
print(X_test_text.shape)

(34705,)
(7439,)
(7438,)


In [30]:
X_train_sequences = tokenizer.texts_to_sequences(X_train_text)
X_val_sequences = tokenizer.texts_to_sequences(X_val_text)
X_test_sequences = tokenizer.texts_to_sequences(X_test_text)

In [31]:
from tensorflow.keras.preprocessing.sequence import pad_sequences

X_train_padded_300 = pad_sequences(
    X_train_sequences,
    maxlen=300,
    padding="post",
    truncating="post"
)

X_val_padded_300 = pad_sequences(
    X_val_sequences,
    maxlen=300,
    padding="post",
    truncating="post"
)

X_test_padded_300 = pad_sequences(
    X_test_sequences,
    maxlen=300,
    padding="post",
    truncating="post"
)

In [32]:
bilstm_sequence = Sequential([
    Input(shape=(300,)),

    Embedding(
        input_dim=vocab_size,
        output_dim=128
    ),
    Bidirectional(
        LSTM(64)
    ),


    Dense(
        64,
        activation="relu"
    ),

    Dense(
        1,
        activation="sigmoid"
    )
])

bilstm_sequence.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=["accuracy"]
)


In [33]:
history_sequence = bilstm_sequence.fit(
    X_train_padded_300,
    y_train,
    validation_data=(X_val_padded_300, y_val),
    epochs=5,
    batch_size=32,
    verbose=1
)

Epoch 1/5
1085/1085 ━━━━━━━━━━━━━━━━━━━━ 177s 162ms/step - accuracy: 0.8288 - loss: 0.3945 - val_accuracy: 0.8840 - val_loss: 0.2941
Epoch 2/5
1085/1085 ━━━━━━━━━━━━━━━━━━━━ 191s 176ms/step - accuracy: 0.9191 - loss: 0.2181 - val_accuracy: 0.8883 - val_loss: 0.3026
Epoch 3/5
1085/1085 ━━━━━━━━━━━━━━━━━━━━ 186s 172ms/step - accuracy: 0.9420 - loss: 0.1570 - val_accuracy: 0.8744 - val_loss: 0.3412
Epoch 4/5
1085/1085 ━━━━━━━━━━━━━━━━━━━━ 195s 180ms/step - accuracy: 0.9627 - loss: 0.1085 - val_accuracy: 0.8833 - val_loss: 0.3463
Epoch 5/5
1085/1085 ━━━━━━━━━━━━━━━━━━━━ 193s 178ms/step - accuracy: 0.9715 - loss: 0.0848 - val_accuracy: 0.8767 - val_loss: 0.3893


In [34]:
evaluate_bilstm(
    bilstm_sequence,
    "BILSTM - Sequence Length 300",
    X_test_padded_300
)
bilstm_sequence.save("models/BILSTM_sequence_300.keras")


BILSTM - Sequence Length 300
----------------------------------------
Accuracy : 0.8729
Precision: 0.8855
Recall   : 0.8578
F1 Score : 0.8714
ROC-AUC  : 0.9416


### Hidden Units

In [35]:
bilstm_hidden = Sequential([
    Input(shape=(sequence_length,)),

    Embedding(
        input_dim=vocab_size,
        output_dim=128
    ),

     Bidirectional(
        LSTM(64)
    ),


    Dense(
        64,
        activation="relu"
    ),

    Dense(
        1,
        activation="sigmoid"
    )
])

bilstm_hidden.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=["accuracy"]
)


In [36]:
history_hidden = bilstm_hidden.fit(
    X_train_padded,
    y_train,
    validation_data=(X_val_padded, y_val),
    epochs=5,
    batch_size=32,
    verbose=1
)


Epoch 1/5
1085/1085 ━━━━━━━━━━━━━━━━━━━━ 155s 142ms/step - accuracy: 0.8364 - loss: 0.3740 - val_accuracy: 0.8810 - val_loss: 0.3032
Epoch 2/5
1085/1085 ━━━━━━━━━━━━━━━━━━━━ 157s 145ms/step - accuracy: 0.9240 - loss: 0.2049 - val_accuracy: 0.8864 - val_loss: 0.2890
Epoch 3/5
1085/1085 ━━━━━━━━━━━━━━━━━━━━ 150s 139ms/step - accuracy: 0.9522 - loss: 0.1360 - val_accuracy: 0.8781 - val_loss: 0.3180
Epoch 4/5
1085/1085 ━━━━━━━━━━━━━━━━━━━━ 140s 129ms/step - accuracy: 0.9677 - loss: 0.0970 - val_accuracy: 0.8700 - val_loss: 0.4303
Epoch 5/5
1085/1085 ━━━━━━━━━━━━━━━━━━━━ 139s 128ms/step - accuracy: 0.9784 - loss: 0.0643 - val_accuracy: 0.8759 - val_loss: 0.4486


In [37]:
evaluate_bilstm(
    bilstm_hidden,
    "BILSTM - Hidden Units 128"
)

bilstm_hidden.save("models/bilstm_hidden_128.keras")


BILSTM - Hidden Units 128
----------------------------------------
Accuracy : 0.8746
Precision: 0.8872
Recall   : 0.8594
F1 Score : 0.8730
ROC-AUC  : 0.9389


### Dropout

In [38]:
bilstm_dropout = Sequential([
    Input(shape=(sequence_length,)),

    Embedding(
        input_dim=vocab_size,
        output_dim=128
    ),

     Bidirectional(
        LSTM(64)
    ),


    Dropout(0.3),

    Dense(
        64,
        activation="relu"
    ),

    Dropout(0.3),

    Dense(
        1,
        activation="sigmoid"
    )
])

bilstm_dropout.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=["accuracy"]
)


In [39]:
history_dropout = bilstm_dropout.fit(
    X_train_padded,
    y_train,
    validation_data=(X_val_padded, y_val),
    epochs=5,
    batch_size=32,
    verbose=1
)


Epoch 1/5
1085/1085 ━━━━━━━━━━━━━━━━━━━━ 134s 122ms/step - accuracy: 0.8240 - loss: 0.4031 - val_accuracy: 0.8783 - val_loss: 0.3035
Epoch 2/5
1085/1085 ━━━━━━━━━━━━━━━━━━━━ 138s 128ms/step - accuracy: 0.9164 - loss: 0.2272 - val_accuracy: 0.8798 - val_loss: 0.2997
Epoch 3/5
1085/1085 ━━━━━━━━━━━━━━━━━━━━ 138s 127ms/step - accuracy: 0.9469 - loss: 0.1517 - val_accuracy: 0.8775 - val_loss: 0.3385
Epoch 4/5
1085/1085 ━━━━━━━━━━━━━━━━━━━━ 138s 127ms/step - accuracy: 0.9664 - loss: 0.0970 - val_accuracy: 0.8684 - val_loss: 0.4336
Epoch 5/5
1085/1085 ━━━━━━━━━━━━━━━━━━━━ 140s 129ms/step - accuracy: 0.9789 - loss: 0.0641 - val_accuracy: 0.8468 - val_loss: 0.4395


In [40]:
evaluate_bilstm(
    bilstm_dropout,
    "BILSTM - Dropout"
)

bilstm_dropout.save("models/bilstm_dropout.keras")


BILSTM - Dropout
----------------------------------------
Accuracy : 0.8446
Precision: 0.8624
Recall   : 0.8213
F1 Score : 0.8414
ROC-AUC  : 0.9251


### Different Optimizer

In [41]:
bilstm_rmsprop = Sequential([
    Input(shape=(sequence_length,)),

    Embedding(
        input_dim=vocab_size,
        output_dim=128
    ),

     Bidirectional(
        LSTM(64)
    ),

    Dense(
        64,
        activation="relu"
    ),

    Dense(
        1,
        activation="sigmoid"
    )
])

bilstm_rmsprop.compile(
    optimizer=RMSprop(learning_rate=0.001),
    loss="binary_crossentropy",
    metrics=["accuracy"]
)


In [42]:

history_rmsprop = bilstm_rmsprop.fit(
    X_train_padded,
    y_train,
    validation_data=(X_val_padded, y_val),
    epochs=5,
    batch_size=32,
    verbose=1
)


Epoch 1/5
1085/1085 ━━━━━━━━━━━━━━━━━━━━ 130s 119ms/step - accuracy: 0.8180 - loss: 0.3969 - val_accuracy: 0.8353 - val_loss: 0.3879
Epoch 2/5
1085/1085 ━━━━━━━━━━━━━━━━━━━━ 131s 121ms/step - accuracy: 0.9022 - loss: 0.2555 - val_accuracy: 0.8558 - val_loss: 0.3722
Epoch 3/5
1085/1085 ━━━━━━━━━━━━━━━━━━━━ 127s 117ms/step - accuracy: 0.9240 - loss: 0.2060 - val_accuracy: 0.8911 - val_loss: 0.2987
Epoch 4/5
1085/1085 ━━━━━━━━━━━━━━━━━━━━ 129s 119ms/step - accuracy: 0.9393 - loss: 0.1701 - val_accuracy: 0.8908 - val_loss: 0.3055
Epoch 5/5
1085/1085 ━━━━━━━━━━━━━━━━━━━━ 128s 118ms/step - accuracy: 0.9555 - loss: 0.1322 - val_accuracy: 0.8785 - val_loss: 0.3353


In [43]:

evaluate_bilstm(
    bilstm_rmsprop,
    "BILSTM - RMSprop"
)

bilstm_rmsprop.save("models/bilstm_rmsprop.keras")


BILSTM - RMSprop
----------------------------------------
Accuracy : 0.8793
Precision: 0.8981
Recall   : 0.8567
F1 Score : 0.8769
ROC-AUC  : 0.9482


### Batch Normalization

In [44]:
bilstm_batchnorm = Sequential([
    Input(shape=(sequence_length,)),

    Embedding(
        input_dim=vocab_size,
        output_dim=128
    ),

     Bidirectional(
        LSTM(64)
    ),


    Dense(
        64,
        activation="relu"
    ),

    BatchNormalization(),

    Dense(
        1,
        activation="sigmoid"
    )
])

bilstm_batchnorm.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=["accuracy"]
)


In [45]:

history_batchnorm = bilstm_batchnorm.fit(
    X_train_padded,
    y_train,
    validation_data=(X_val_padded, y_val),
    epochs=5,
    batch_size=32,
    verbose=1
)


Epoch 1/5
1085/1085 ━━━━━━━━━━━━━━━━━━━━ 134s 122ms/step - accuracy: 0.8361 - loss: 0.3768 - val_accuracy: 0.8353 - val_loss: 0.3980
Epoch 2/5
1085/1085 ━━━━━━━━━━━━━━━━━━━━ 137s 127ms/step - accuracy: 0.9160 - loss: 0.2206 - val_accuracy: 0.8843 - val_loss: 0.3145
Epoch 3/5
1085/1085 ━━━━━━━━━━━━━━━━━━━━ 142s 131ms/step - accuracy: 0.9518 - loss: 0.1349 - val_accuracy: 0.8783 - val_loss: 0.3513
Epoch 4/5
1085/1085 ━━━━━━━━━━━━━━━━━━━━ 139s 128ms/step - accuracy: 0.9706 - loss: 0.0863 - val_accuracy: 0.8632 - val_loss: 0.4027
Epoch 5/5
1085/1085 ━━━━━━━━━━━━━━━━━━━━ 139s 128ms/step - accuracy: 0.9801 - loss: 0.0600 - val_accuracy: 0.8707 - val_loss: 0.5121


In [46]:

evaluate_bilstm(
    bilstm_batchnorm,
    "BILSTM - Batch Normalization"
)

bilstm_batchnorm.save("models/bilstm_batchnorm.keras")


BILSTM - Batch Normalization
----------------------------------------
Accuracy : 0.8705
Precision: 0.8837
Recall   : 0.8545
F1 Score : 0.8689
ROC-AUC  : 0.9343


### Learning Rate

In [47]:
bilstm_learning_rate = Sequential([
    Input(shape=(sequence_length,)),

    Embedding(
        input_dim=vocab_size,
        output_dim=128
    ),

    Bidirectional(
        LSTM(64)
    ),

    Dense(
        64,
        activation="relu"
    ),

    Dense(
        1,
        activation="sigmoid"
    )
])

bilstm_learning_rate.compile(
    optimizer=Adam(learning_rate=0.0001),
    loss="binary_crossentropy",
    metrics=["accuracy"]
)


In [48]:

history_learning_rate = bilstm_learning_rate.fit(
    X_train_padded,
    y_train,
    validation_data=(X_val_padded, y_val),
    epochs=5,
    batch_size=32,
    verbose=1
)



Epoch 1/5
1085/1085 ━━━━━━━━━━━━━━━━━━━━ 138s 126ms/step - accuracy: 0.8012 - loss: 0.4122 - val_accuracy: 0.8908 - val_loss: 0.2743
Epoch 2/5
1085/1085 ━━━━━━━━━━━━━━━━━━━━ 143s 131ms/step - accuracy: 0.9223 - loss: 0.2108 - val_accuracy: 0.8921 - val_loss: 0.2664
Epoch 3/5
1085/1085 ━━━━━━━━━━━━━━━━━━━━ 142s 131ms/step - accuracy: 0.9470 - loss: 0.1503 - val_accuracy: 0.8921 - val_loss: 0.3022
Epoch 4/5
1085/1085 ━━━━━━━━━━━━━━━━━━━━ 140s 129ms/step - accuracy: 0.9630 - loss: 0.1123 - val_accuracy: 0.8840 - val_loss: 0.3546
Epoch 5/5
1085/1085 ━━━━━━━━━━━━━━━━━━━━ 140s 129ms/step - accuracy: 0.9729 - loss: 0.0863 - val_accuracy: 0.8820 - val_loss: 0.3902


In [49]:
evaluate_bilstm(
    bilstm_learning_rate,
    "BILSTM - Learning Rate 0.0001"
)

bilstm_learning_rate.save("models/bilstm_learning_rate.keras")


BILSTM - Learning Rate 0.0001
----------------------------------------
Accuracy : 0.8751
Precision: 0.8567
Recall   : 0.9020
F1 Score : 0.8788
ROC-AUC  : 0.9464


### Batch Size

In [52]:
bilstm_batch_size = Sequential([
    Input(shape=(sequence_length,)),

    Embedding(
        input_dim=vocab_size,
        output_dim=128
    ),

    Bidirectional(
        LSTM(64)
    ),

    Dense(
        64,
        activation="relu"
    ),

    Dense(
        1,
        activation="sigmoid"
    )
])

bilstm_batch_size.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=["accuracy"]
)


In [53]:

history_batch_size = bilstm_batch_size.fit(
    X_train_padded,
    y_train,
    validation_data=(X_val_padded, y_val),
    epochs=5,
    batch_size=64,
    verbose=1
)


Epoch 1/5
543/543 ━━━━━━━━━━━━━━━━━━━━ 94s 170ms/step - accuracy: 0.8405 - loss: 0.3646 - val_accuracy: 0.8813 - val_loss: 0.2877
Epoch 2/5
543/543 ━━━━━━━━━━━━━━━━━━━━ 96s 177ms/step - accuracy: 0.9269 - loss: 0.1981 - val_accuracy: 0.8750 - val_loss: 0.3330
Epoch 3/5
543/543 ━━━━━━━━━━━━━━━━━━━━ 96s 178ms/step - accuracy: 0.9521 - loss: 0.1372 - val_accuracy: 0.8853 - val_loss: 0.3335
Epoch 4/5
543/543 ━━━━━━━━━━━━━━━━━━━━ 96s 177ms/step - accuracy: 0.9674 - loss: 0.0955 - val_accuracy: 0.8804 - val_loss: 0.4002
Epoch 5/5
543/543 ━━━━━━━━━━━━━━━━━━━━ 97s 179ms/step - accuracy: 0.9762 - loss: 0.0711 - val_accuracy: 0.8781 - val_loss: 0.3839


In [54]:
evaluate_bilstm(
    bilstm_batch_size,
    "BILSTM - Batch Size 64"
)

bilstm_batch_size.save("models/GRU_batch_size_64.keras")


BILSTM - Batch Size 64
----------------------------------------
Accuracy : 0.8716
Precision: 0.8841
Recall   : 0.8564
F1 Score : 0.8701
ROC-AUC  : 0.9382


### Early Stopping

In [55]:
bilstm_early_stopping = Sequential([
    Input(shape=(sequence_length,)),

    Embedding(
        input_dim=vocab_size,
        output_dim=128
    ),

    Bidirectional(
        LSTM(64)
    ),

    Dense(
        64,
        activation="relu"
    ),

    Dense(
        1,
        activation="sigmoid"
    )
])

bilstm_early_stopping.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

early_stopping = EarlyStopping(
    monitor="val_loss",
    patience=2,
    restore_best_weights=True
)


In [56]:

history_early_stopping = bilstm_early_stopping.fit(
    X_train_padded,
    y_train,
    validation_data=(X_val_padded, y_val),
    epochs=10,
    batch_size=32,
    callbacks=[early_stopping],
    verbose=1
)



Epoch 1/10
1085/1085 ━━━━━━━━━━━━━━━━━━━━ 135s 124ms/step - accuracy: 0.8388 - loss: 0.3673 - val_accuracy: 0.8804 - val_loss: 0.2959
Epoch 2/10
1085/1085 ━━━━━━━━━━━━━━━━━━━━ 141s 130ms/step - accuracy: 0.9253 - loss: 0.2029 - val_accuracy: 0.8629 - val_loss: 0.3311
Epoch 3/10
1085/1085 ━━━━━━━━━━━━━━━━━━━━ 143s 132ms/step - accuracy: 0.9592 - loss: 0.1182 - val_accuracy: 0.8767 - val_loss: 0.3905


In [57]:
evaluate_bilstm(
    bilstm_early_stopping,
    "BILSTM - Early Stopping"
)

bilstm_early_stopping.save(
    "models/bilstm_early_stopping.keras"
)


BILSTM - Early Stopping
----------------------------------------
Accuracy : 0.8775
Precision: 0.8371
Recall   : 0.9387
F1 Score : 0.8850
ROC-AUC  : 0.9506


### Learning Rate Scheduling

In [58]:
bilstm_scheduler = Sequential([
    Input(shape=(sequence_length,)),

    Embedding(
        input_dim=vocab_size,
        output_dim=128
    ),

    Bidirectional(
        LSTM(64)
    ),

    Dense(
        64,
        activation="relu"
    ),

    Dense(
        1,
        activation="sigmoid"
    )
])

bilstm_scheduler.compile(
    optimizer=Adam(learning_rate=0.001),
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

reduce_lr = ReduceLROnPlateau(
    monitor="val_loss",
    factor=0.5,
    patience=1,
    min_lr=1e-6
)


In [59]:

history_scheduler = bilstm_scheduler.fit(
    X_train_padded,
    y_train,
    validation_data=(X_val_padded, y_val),
    epochs=5,
    batch_size=32,
    callbacks=[reduce_lr],
    verbose=1
)


Epoch 1/5
1085/1085 ━━━━━━━━━━━━━━━━━━━━ 137s 125ms/step - accuracy: 0.8419 - loss: 0.3656 - val_accuracy: 0.8857 - val_loss: 0.2814 - learning_rate: 0.0010
Epoch 2/5
1085/1085 ━━━━━━━━━━━━━━━━━━━━ 141s 130ms/step - accuracy: 0.9237 - loss: 0.2068 - val_accuracy: 0.8856 - val_loss: 0.2900 - learning_rate: 0.0010
Epoch 3/5
1085/1085 ━━━━━━━━━━━━━━━━━━━━ 143s 131ms/step - accuracy: 0.9659 - loss: 0.1016 - val_accuracy: 0.8794 - val_loss: 0.3466 - learning_rate: 5.0000e-04
Epoch 4/5
1085/1085 ━━━━━━━━━━━━━━━━━━━━ 146s 134ms/step - accuracy: 0.9856 - loss: 0.0481 - val_accuracy: 0.8855 - val_loss: 0.4265 - learning_rate: 2.5000e-04
Epoch 5/5
1085/1085 ━━━━━━━━━━━━━━━━━━━━ 142s 131ms/step - accuracy: 0.9940 - loss: 0.0243 - val_accuracy: 0.8773 - val_loss: 0.5060 - learning_rate: 1.2500e-04


In [60]:

evaluate_bilstm(
    bilstm_scheduler,
    "BILSTM - Learning Rate Scheduling"
)

bilstm_scheduler.save(
    "models/BILSTM_learning_rate_scheduler.keras"
)


BILSTM - Learning Rate Scheduling
----------------------------------------
Accuracy : 0.8746
Precision: 0.8652
Recall   : 0.8886
F1 Score : 0.8767
ROC-AUC  : 0.9404


### Recurrent Dropout

In [63]:
bilstm_recurrent_dropout = Sequential([
    Input(shape=(sequence_length,)),

    Embedding(
        input_dim=vocab_size,
        output_dim=128
    ),

    Bidirectional(
        LSTM(64,recurrent_dropout=0.3)
    ),

    Dense(
        64,
        activation="relu"
    ),

    Dense(
        1,
        activation="sigmoid"
    )
])

bilstm_recurrent_dropout.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=["accuracy"]
)



In [65]:
history_recurrent_dropout = bilstm_recurrent_dropout.fit(
    X_train_padded,
    y_train,
    validation_data=(X_val_padded, y_val),
    epochs=5,
    batch_size=32,
    verbose=1
)


Epoch 1/5
1085/1085 ━━━━━━━━━━━━━━━━━━━━ 215s 199ms/step - accuracy: 0.8377 - loss: 0.3883 - val_accuracy: 0.8720 - val_loss: 0.3173
Epoch 2/5
1085/1085 ━━━━━━━━━━━━━━━━━━━━ 212s 195ms/step - accuracy: 0.9004 - loss: 0.2579 - val_accuracy: 0.8801 - val_loss: 0.3031
Epoch 3/5
1085/1085 ━━━━━━━━━━━━━━━━━━━━ 221s 203ms/step - accuracy: 0.9300 - loss: 0.1885 - val_accuracy: 0.8837 - val_loss: 0.3352
Epoch 4/5
1085/1085 ━━━━━━━━━━━━━━━━━━━━ 224s 206ms/step - accuracy: 0.9463 - loss: 0.1475 - val_accuracy: 0.8779 - val_loss: 0.3633
Epoch 5/5
1085/1085 ━━━━━━━━━━━━━━━━━━━━ 212s 196ms/step - accuracy: 0.9648 - loss: 0.1008 - val_accuracy: 0.8761 - val_loss: 0.4081


In [66]:

evaluate_bilstm(
    bilstm_recurrent_dropout,
    "BILSM - Recurrent Dropout"
)

bilstm_recurrent_dropout.save(
    "models/bilstm_recurrent_dropout.keras"
)


BILSM - Recurrent Dropout
----------------------------------------
Accuracy : 0.8723
Precision: 0.8849
Recall   : 0.8570
F1 Score : 0.8707
ROC-AUC  : 0.9378


In [68]:
bilstm_results_df = pd.DataFrame(bilstm_results)

bilstm_results_df

,Experiment,Accuracy,Precision,Recall,F1 Score,ROC-AUC
0,BILSTM - Baseline,0.867169,0.873063,0.860434,0.866703,0.936112
1,BILSTM - Baseline,0.867169,0.873063,0.860434,0.866703,0.936112
2,BILSTM - Embedding 256,0.865421,0.839801,0.904366,0.870889,0.937141
3,BILSTM - Sequence Length 300,0.866765,0.853533,0.886686,0.869794,0.930051
4,BILSTM - Embedding 256,0.865555,0.853192,0.884275,0.868456,0.929578
5,BILSTM - Sequence Length 300,0.872950,0.885509,0.857755,0.871411,0.941650
6,BILSTM - Hidden Units 128,0.874563,0.887168,0.859362,0.873044,0.938896
7,BILSTM - Dropout,0.844582,0.862447,0.821323,0.841383,0.925061
8,BILSTM - RMSprop,0.879269,0.898062,0.856684,0.876885,0.948213
9,BILSTM - Batch Normalization,0.870530,0.883657,0.854541,0.868855,0.934338


In [69]:
bilstm_results_df.to_csv("models/bilstm_comparison_table.csv",index=False)